# Training of the parking spot detection

Trains the `ParkingNet` (see `parking/perception/network.py`) on parking spots annotated in CVAT.

Expected data: one or more datasets, each with CVAT exports **"CVAT for images 1.1"** (`annotations*.xml`) and one folder of images per rosbag:

```text
data/dataset_v5/
├── annotations.xml                  (one export per CVAT task, name annotations*.xml)
└── rosbag2_2026_10_28-13_32_53/     (folder of the share, one per rosbag)
    ├── rosbag2_2026_10_28-13_32_53_000000.png
    └── ...
```

The trained model is saved to `models/parking_net.pt` and loaded by the `parking_perception_node`.

## 1. Import Libraries

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import tqdm
from torch import optim
from torch.utils.data import DataLoader

# Import the parking package without sourcing the ROS workspace
PACKAGE_DIR = Path.cwd().parent
sys.path.insert(0, str(PACKAGE_DIR))

from parking.perception.network import ParkingNet, decode  # noqa: E402
from parking.training.dataset import (  # noqa: E402
    ParkingSpotDataset,
    bag_name,
    cvat_labels,
    load_cvat_xml,
    split_by_blocks,
)
from parking.training.loss import ParkingLoss  # noqa: E402

print("PyTorch version:", torch.__version__)
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    print("Using Apple GPU (MPS).")
    device = torch.device("mps")
else:
    print("No GPU found, using CPU instead.")
    device = torch.device("cpu")

## 2. Data Loading

The classes are the labels of the CVAT export, in the order defined in CVAT. They are stored in the model, so the node and the CVAT function use the same classes.

All datasets in `DATA_DIRS` are combined. They have to come from the same CVAT project (same labels).

Consecutive frames of a rosbag are almost identical. Therefore every rosbag is cut into blocks of `BLOCK_SIZE` consecutive frames and a random part of the blocks **of every rosbag** is used for the validation. So every rosbag and maneuver (driving past, parking in) is in the training and in the validation, and nearly identical neighbor frames are not split between them.

In [ ]:
DATA_DIRS = [
    PACKAGE_DIR / "data" / "dataset_v4",  # Driving past the parking spots
    PACKAGE_DIR / "data" / "dataset_v5",  # Also parking into two 90 degree spots
]
INPUT_SIZE = (400, 320)  # Network input (width, height), divisible by 16
VAL_FRACTION = 0.2
BLOCK_SIZE = 10  # Consecutive images per block (10 images = 50 frames of the rosbag)
BATCH_SIZE = 8
SEED = 42

XML_FILES = {d: sorted(d.glob("annotations*.xml")) for d in DATA_DIRS}
for data_dir, files in XML_FILES.items():
    print(f"{data_dir.name}:", [f.name for f in files])
    if not files:
        raise FileNotFoundError(f"No annotations*.xml in {data_dir}")

# All tasks have to belong to the same CVAT project, i.e. have the same labels
all_files = [f for files in XML_FILES.values() for f in files]
CLASSES = cvat_labels(all_files[0])
for xml_file in all_files[1:]:
    if cvat_labels(xml_file) != CLASSES:
        raise ValueError(f"Different labels in {xml_file}")
print("Classes:", CLASSES)

samples = [
    s
    for data_dir, files in XML_FILES.items()
    for f in files
    for s in load_cvat_xml(f, data_dir, CLASSES)
]
print(f"{len(samples)} images, {sum(len(s.spots) for s in samples)} parking spots")
for class_id, name in enumerate(CLASSES):
    count = sum(a.class_id == class_id for s in samples for a in s.spots)
    print(f"  {name}: {count}")

# Every image may only appear once (e.g. the same rosbag in two datasets)
names = [s.image_path.name for s in samples]
duplicates = {n for n in names if names.count(n) > 1}
if duplicates:
    raise ValueError(f"Images in several datasets, e.g. {sorted(duplicates)[:3]}")

train_samples, val_samples = split_by_blocks(samples, VAL_FRACTION, BLOCK_SIZE, SEED)
for bag in sorted({bag_name(s) for s in samples}):
    num_train = sum(bag_name(s) == bag for s in train_samples)
    num_val = sum(bag_name(s) == bag for s in val_samples)
    print(f"  {bag}: train {num_train}, val {num_val}")
print(f"Train: {len(train_samples)} images, validation: {len(val_samples)} images")

train_dataset = ParkingSpotDataset(
    train_samples, len(CLASSES), INPUT_SIZE, augment=True
)
val_dataset = ParkingSpotDataset(val_samples, len(CLASSES), INPUT_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Show some training images with their target heatmap (all classes combined)
batch = next(iter(train_loader))
num_images = min(4, len(batch["image"]))
fig, axes = plt.subplots(2, num_images, figsize=(5 * num_images, 8), squeeze=False)

for idx in range(num_images):
    axes[0, idx].imshow(batch["image"][idx, 0], cmap="gray")
    axes[0, idx].set_title("Input (augmented)")
    axes[1, idx].imshow(batch["heatmap"][idx].max(dim=0).values, cmap="hot")
    axes[1, idx].set_title("Target heatmap")
    for ax in axes[:, idx]:
        ax.set_xticks([])
        ax.set_yticks([])

## 3. Model Definition, Loss Function, and Optimizer

In [ ]:
WIDTH = 16  # Channels of the first layer, increase for a bigger model
LEARNING_RATE = 1e-3

model = ParkingNet(num_classes=len(CLASSES), width=WIDTH).to(device)
criterion = ParkingLoss(corner_weight=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Training Loop

In [ ]:
def to_device(batch: dict) -> dict:
    return {key: value.to(device) for key, value in batch.items()}


def train(
    data_loader: DataLoader,
    model: ParkingNet,
    criterion: ParkingLoss,
    optimizer: optim.Optimizer,
) -> float:
    model.train()
    total_loss = 0.0

    for cpu_batch in tqdm.tqdm(data_loader, total=len(data_loader)):
        batch = to_device(cpu_batch)

        output = model(batch["image"])  # forward pass
        loss, _ = criterion(output, batch)  # compute loss

        optimizer.zero_grad()
        loss.backward()  # backward pass
        optimizer.step()  # update weights

        total_loss += loss.item()

    return total_loss / len(data_loader)


def validate(
    data_loader: DataLoader, model: ParkingNet, criterion: ParkingLoss
) -> float:
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for cpu_batch in data_loader:
            batch = to_device(cpu_batch)
            loss, _ = criterion(model(batch["image"]), batch)
            total_loss += loss.item()

    return total_loss / len(data_loader)

In [ ]:
NUM_EPOCHS = 50
MODEL_PATH = PACKAGE_DIR / "models" / "parking_net.pt"

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
train_losses = []
val_losses = []
best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    train_loss = train(train_loader, model, criterion, optimizer)
    val_loss = validate(val_loader, model, criterion)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] -> "
        f"Train loss: {train_loss:.4f}, Val loss: {val_loss:.4f}"
    )

    # Save the best model for the parking_perception_node
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "state_dict": model.state_dict(),
                "classes": CLASSES,
                "input_size": INPUT_SIZE,
                "width": WIDTH,
            },
            MODEL_PATH,
        )
        print(f"  Saved best model to {MODEL_PATH}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.yscale("log")
plt.title("Loss over Epochs")
plt.legend()
plt.show()

## 5. Testing

Draws the predictions of the best model (green, blue dot: first corner = bottom-left) and the annotations (red) into validation images.

In [ ]:
SCORE_THRESHOLD = 0.3

checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

num_images = min(6, len(val_dataset))
fig, axes = plt.subplots(
    (num_images + 2) // 3, 3, figsize=(18, 5 * ((num_images + 2) // 3)), squeeze=False
)

for ax in axes.flat:
    ax.axis("off")

for idx in range(num_images):
    sample = val_dataset[idx]
    with torch.no_grad():
        heatmap, corners = model(sample["image"][None].to(device))
    detections = decode(heatmap[0], corners[0], SCORE_THRESHOLD)

    ax = axes.flat[idx]
    ax.imshow(sample["image"][0], cmap="gray")

    scale = torch.tensor(INPUT_SIZE) / torch.tensor(
        plt.imread(val_samples[idx].image_path).shape[1::-1]
    )
    for annotation in val_samples[idx].spots:
        gt = annotation.corners * scale.numpy()
        ax.fill(gt[:, 0], gt[:, 1], edgecolor="red", fill=False, linewidth=2)

    for class_id, score, spot_corners in detections:
        ax.fill(
            spot_corners[:, 0],
            spot_corners[:, 1],
            edgecolor="lime",
            fill=False,
            linewidth=2,
        )
        ax.plot(*spot_corners[0], "o", color="blue")
        ax.text(
            *spot_corners.mean(axis=0), f"{CLASSES[class_id]} {score:.2f}", color="lime"
        )